# Step 4: Hub labels and modeling table

Hub = top 20% centrality within chromosome, per stage. Stable = hub in 3-4 stages, Transient = hub in 1-2, 0-stage genes dropped. Merges with the histone features into one table for the model notebook.

## Config

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
PROC = ROOT / "data" / "processed"
GENE_COORDS = PROC / "gene_coordinates.csv"
EIGEN_CSV = PROC / "eigenvector_features.csv"
HISTONE_CSV = PROC / "histone_features.csv"

OUT = PROC / "modeling_table.csv"
STAGES = ["C1-8", "C9-13", "C14E", "C14L"]
HUB_QUANTILE = 0.80

for p in [GENE_COORDS, EIGEN_CSV, HISTONE_CSV]: 
    assert p.exists(), f"missing {p}"

## Load and merge chrom onto the centrality table

In [10]:
coords = pd.read_csv(GENE_COORDS)[["gene_name", "chrom"]]
eigen = pd.read_csv(EIGEN_CSV)
histone = pd.read_csv(HISTONE_CSV)

eigen = eigen.merge(coords, on="gene_name", how="left", suffixes=("", "_coords"))
print(f"{len(eigen)} genes, {len(histone)} genes with histone features")

16846 genes, 16846 genes with histone features


## Hub calls per stage

In [11]:
# cutoff is per chromosome, per stage. NaN centrality just never clears the
# cutoff, so a gene with no Hi-C bin for a stage is never called a hub there

eigen["high_connectivity_count"] = 0
for stage in STAGES:
    col = f"centrality_{stage}"
    cutoff = eigen.groupby("chrom")[col].transform(lambda s: s.quantile(HUB_QUANTILE))
    eigen[f"{stage}_is_hub"] = (eigen[col] > cutoff).astype(int)
    eigen["high_connectivity_count"] += eigen[f"{stage}_is_hub"]

print(eigen["high_connectivity_count"].value_counts().sort_index())

high_connectivity_count
0    12337
1      750
2      894
3      653
4     2212
Name: count, dtype: int64


## Stable/Transient label

In [12]:
def label(count):
    if count >= 3:
        return 1  # Stable
    if count in (1, 2):
        return 0  # Transient
    return np.nan  # never a hub, dropped

eigen["target"] = eigen["high_connectivity_count"].apply(label)
labeled = eigen.dropna(subset=["target"]).copy()
labeled["target"] = labeled["target"].astype(int)

print(labeled["target"].value_counts().rename({0: "Transient", 1: "Stable"}))

target
Stable       2865
Transient    1644
Name: count, dtype: int64


## Merge with histone features, save

In [13]:
hub_cols = ["gene_name", "target", "high_connectivity_count"] + [f"{s}_is_hub" for s in STAGES]
modeling = labeled[hub_cols].merge(histone, on="gene_name", how="inner")

print(f"modeling table: {modeling.shape[0]} genes x {modeling.shape[1]} columns")
print(f"dropped {len(labeled) - len(modeling)} genes with no histone row")

modeling.to_csv(OUT, index=False)
print(f"saved -> {OUT.relative_to(ROOT)}")

modeling table: 4509 genes x 39 columns
dropped 0 genes with no histone row
saved -> data\processed\modeling_table.csv
